# 00 Theory And Conventions

I start the study by pointing the notebook workflow at the written theory and metric glossary. The science should be defined before the code measures it, so this notebook checks that the backbone documents are present and indexes the sections I expect future-me to read first.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

from vbb_study import setup_study, vbb_style

PATHS = setup_study.bootstrap(Path.cwd())
DOCS = PATHS["docs"]
PUB_OUT = PATHS["outputs"]
(PUB_OUT / "csv" / "publication_study").mkdir(parents=True, exist_ok=True)

## Editable Notebook Controls

<!-- STAGE88: editable controls -->

This cell exposes the intended user-editable controls for exploratory runs. The locked stage logic below is preserved: changing these controls is for local investigation unless the notebook explicitly wires a value into a regenerated canonical output. Keep QA, caveats, and fail/marginal labels visible. For fast beam-to-sample exploration use the quicklook notebook; for publication-grade outputs use the locked stage runner.


In [2]:
# STAGE88: visible editable controls for exploratory notebook use.
from vbb_study.publication import notebook_controls as nb_controls

NOTEBOOK_CONTROLS = nb_controls.make_notebook_controls(stage='overview')
try:
    display(nb_controls.describe_controls(NOTEBOOK_CONTROLS))
except NameError:
    print(nb_controls.describe_controls(NOTEBOOK_CONTROLS).to_string(index=False))


,control,value
0,stage,overview
1,run_mode,quick_preview
2,save_outputs,False
3,use_canonical_outputs,True
4,allow_publication_export,False
5,notes,Edit for exploration; keep QA labels/caveats v...


## Document Index

I keep the derivations and definitions in markdown because they are easier to diff and cite than notebook prose. The notebook records the document set as a reproducibility artefact.

In [3]:
docs = [
    ("Theory derivation", DOCS / "00_theory.md"),
    ("Metric conventions", DOCS / "01_conventions.md"),
    ("Validation record", DOCS / "02_validation.md"),
    ("Materials proxy", DOCS / "03_materials_application.md"),
]
rows = []
for label, path in docs:
    text = path.read_text(encoding="utf-8")
    headings = [line.strip("# ") for line in text.splitlines() if line.startswith("## ")]
    rows.append({
        "document": label,
        "path": str(path.relative_to(PATHS["root"])),
        "bytes": path.stat().st_size,
        "section_count": len(headings),
        "first_sections": "; ".join(headings[:4]),
    })
index = pd.DataFrame(rows)
index.to_csv(PUB_OUT / "csv" / "publication_study" / vbb_style.csv_name(0, "theory", "document_index"), index=False)
display(index)

,document,path,bytes,section_count,first_sections
0,Theory derivation,Publication_Study/docs/00_theory.md,8138,11,1. True Bessel-Gauss target field; 1a. Conical...
1,Metric conventions,Publication_Study/docs/01_conventions.md,7178,14,Vortex Ring Radius; J0 First-Zero Radius; HWHM...
2,Validation record,Publication_Study/docs/02_validation.md,3534,3,Current Validation Table; Validation Notes; No...
3,Materials proxy,Publication_Study/docs/03_materials_applicatio...,2828,5,Status; Planning Proxy Warning; What The Layer...


In [4]:
display(Markdown((DOCS / "01_conventions.md").read_text(encoding="utf-8").split("## Engine Metric Key Glossary")[0]))

# Metric Conventions Glossary

Single source of truth for reported scalar-core metrics. Each entry gives the
meaning, units, plane/medium, and implementing function.

## Vortex Ring Radius

| | |
|---|---|
| Symbol | `ring_radius_m`, `vortex_main_ring_radius_m` |
| Formula | `jnp_zeros(abs(ell), 1)[0] / k_r`, the first zero of `J'_ell` |
| Units | metres in code, micrometres in tables |
| Plane / medium | transverse, sample medium |
| Implementing function | `bessel_twin_core.compute_design_from_targets`; measured by `vbb_study.vbb_metrics.peak_plane_radial_metrics` |
| Notes | Defined for `abs(ell) > 0`. The vortex center is dark, so the bright feature is the annular ring, not a central core. |

## J0 First-Zero Radius

| | |
|---|---|
| Symbol | `core_first_zero_radius_m`, `equivalent_l0_first_zero_radius_m` |
| Formula | `2.405 / k_r`, the first zero of `J0(k_r r)` |
| Units | metres in code, micrometres in tables |
| Plane / medium | transverse design scale, sample medium |
| Implementing function | `bessel_twin_core.compute_design_from_targets`; radial metrics also report `core_first_zero_radius_m` |
| Notes | For `ell = 0`, the central maximum is at `r = 0`; this radius is the first dark ring. For `ell > 0`, it is only an equivalent l0 scale and is not the vortex ring radius. |

## HWHM Core Radius

| | |
|---|---|
| Symbol | `core_hwhm_radius_m`, `core_hwhm_diameter_m` |
| Formula | Outer radius where the `ell = 0` azimuthal-average intensity falls to 50% of the central maximum |
| Units | metres in code, micrometres in tables |
| Plane / medium | transverse, sample medium |
| Implementing function | `vbb_study.vbb_metrics.peak_plane_radial_metrics` |
| Notes | This is the measured bright-core size for `ell = 0`. Legacy `core_radius_m` remains for compatibility and is disambiguated by `core_radius_definition`; new code should prefer `core_hwhm_radius_m` or `core_first_zero_radius_m`. |

## Feature Radius And Diameter

| | |
|---|---|
| Symbol | `feature_radius_m`, `feature_diameter_m` |
| Formula | `core_hwhm_radius_m` for `ell = 0`; `ring_radius_m` for `abs(ell) > 0` |
| Units | metres in code, micrometres in tables |
| Plane / medium | transverse, sample medium |
| Implementing function | `vbb_study.vbb_metrics.radial_feature_metrics` |
| Notes | Use this when a plot or CSV needs one shape-size column that works for both central-core and vortex-ring cases. |

## Equivalent l0 Target Diameter

| | |
|---|---|
| Symbol | `target_core_diameter_m`, `target_equivalent_l0_core_diameter_m` |
| Formula | `target_core_diameter_m = 2 * 2.405 / k_r` in the current compatibility mode |
| Units | metres in code, micrometres in tables |
| Plane / medium | inverse-design scale, sample medium |
| Implementing function | `bessel_twin_core.compute_design_from_targets` |
| Notes | Warning: `target_core_diameter_m` currently means `target_scale_definition = "equivalent_l0_first_zero_diameter"`. For vortex beams, the actual bright ring diameter is `vortex_main_ring_diameter_m`, not `target_core_diameter_m`. |

## Ring Width

| | |
|---|---|
| Symbol | `ring_width_m` |
| Formula | Radial half-width at half maximum of the azimuthal-average profile around the vortex ring: `r_half_outer_m - r_half_inner_m` |
| Units | metres in code, micrometres in tables |
| Plane / medium | transverse, sample medium |
| Implementing function | `vbb_study.vbb_metrics.peak_plane_radial_metrics` |

## Canonical Bessel Zone

| | |
|---|---|
| Symbol | `bessel_zone_um`, also exported as `canonical_zone_um` |
| Formula | FWHM of the axial peak-intensity trace: z range where `max_xy I(z) >= 0.5 * max_z(max_xy I)` |
| Units | micrometres |
| Plane / medium | axial, same medium as the volume scan |
| Implementing function | `bessel_twin_core.bessel_zone_metrics(z, peak, level=0.5)` |
| Notes | This is the canonical single-observable zone metric. It is shorter than the geometric `z_max` for finite Gaussian apertures. |

## Strict Bessel Region

| | |
|---|---|
| Symbol | `strict_bessel_region_um`, also exported as `bessel_region_um` |
| Formula | Contiguous intersection of axial-peak threshold, fixed-bucket feature-power threshold, and ring/core radius-stability threshold |
| Units | micrometres |
| Plane / medium | axial, same medium as the volume scan |
| Implementing function | `bessel_twin_core.bessel_region_metrics` |
| Notes | This is the conservative useful region for fabrication-planning reports. It must not be silently replaced by the canonical FWHM zone. |

## Side-To-Core Peak Ratio

| | |
|---|---|
| Symbol | `side_to_core_peak_ratio` |
| Formula | Brightest excluded side-lobe peak divided by the bright feature peak |
| Units | dimensionless |
| Plane / medium | transverse, peak intensity plane |
| Implementing function | `bessel_twin_core.fluence_metrics` |
| Notes | For vortices, the bright feature is the annular HWHM bucket. For `ell = 0`, it is the HWHM core disk. Values above 0.5 indicate significant side-lobe contamination; values above 0.8 are marginal. |

## Fluence

| | |
|---|---|
| Symbol | `F` |
| Formula | `F = E_pulse * I(x, y) / integral(I dA)`, converted from J/m^2 to J/cm^2 |
| Units | J/cm^2 |
| Plane / medium | one transverse XY plane |
| Implementing function | `bessel_twin_core.fluence_from_intensity` or `vbb_study.vbb_materials.fluence_from_intensity` |
| Notes | Energy-conserving on a single plane. Centerline XZ fluence plots are planning proxies, not calibrated material-response predictions. |

## Incubated Threshold

| | |
|---|---|
| Symbol | `F_th,N` |
| Formula | `F_th,N = F_th,1 * N_eff ** (S - 1)` |
| Units | J/cm^2 |
| Plane / medium | material proxy |
| Implementing function | `vbb_study.vbb_materials_study.incubated_threshold`; `vbb_study.vbb_materials.incubated_threshold_J_cm2` |
| Notes | Threshold values are planning proxies until calibrated by experiment. |

## Sampling Validity

| | |
|---|---|
| Symbol | `sampling_valid`, `qa_status`, `phase_sampling_label` |
| Criteria | Feature size, radial period, axial sampling, and SLM/propagation Nyquist checks |
| Units | dimensionless counts or labels |
| Implementing function | `bessel_twin_core.sampling_report`; `vbb_study.vbb_regime.sampling_validity` |
| Notes | Marginal sampling is diagnostic; failed sampling should not be treated as a publication-quality result. |

## First-Order Geometry Validity

| | |
|---|---|
| Symbol | `first_order_geometry_valid` |
| Formula | `cone_lpmm + filter_lpmm < carrier_lpmm` with a small bin margin |
| Units | lp/mm |
| Implementing function | `bessel_twin_core.first_order_filter_geometry` |
| Notes | If false, holographic diffraction orders overlap. The physical axicon route is not constrained by this first-order filtering geometry. |

## Energy Budget Symbols

| symbol | key | units | formula |
|---|---|---|---|
| `E_in` | `pulse_energy_in_J` | uJ | laser output per pulse |
| `E_surface_air` | `pulse_energy_at_surface_air_J` | uJ | `E_in` times pre-surface transmissions |
| `E_sample` | `pulse_energy_at_sample_J` | uJ | `E_surface_air` times surface transmission |
| `T_total` | `total_transmission` | dimensionless | product of modeled transmissions |
